# Why Lasso creates sparsity: the one-input formula

In [1]:
import warnings
import numpy as np
from sklearn.datasets import make_regression
from sklearn.linear_model import Lasso
warnings.simplefilter("ignore")

## The three cases

In [2]:
def lasso_slope(S, D, lam):
    if S > lam:            # slope positive
        return (S - lam) / D
    if S < -lam:           # slope negative
        return (S + lam) / D
    return 0.0             # -lam <= S <= lam

def ridge_slope(S, D, lam):
    return S / (D + lam)

for lam in [0, 25, 50, 100, 150, 1000]:
    print(f"lambda {lam:5d}: Lasso {lasso_slope(100, 50, lam):5.2f}   Ridge {ridge_slope(100, 50, lam):.3f}")

lambda     0: Lasso  2.00   Ridge 2.000
lambda    25: Lasso  1.50   Ridge 1.333
lambda    50: Lasso  1.00   Ridge 1.000
lambda   100: Lasso  0.00   Ridge 0.667
lambda   150: Lasso  0.00   Ridge 0.500
lambda  1000: Lasso  0.00   Ridge 0.095


## Check against scikit-learn on the 100-point example

scikit-learn's Lasso minimises (1/2n) * squared error + alpha * |m|, so alpha = lambda / n.

In [3]:
X, y = make_regression(n_samples=100, n_features=1, n_informative=1, noise=20, random_state=13)
x = X.ravel()
S = ((x - x.mean()) * (y - y.mean())).sum()
D = ((x - x.mean()) ** 2).sum()
print(f"S = {S:.2f}, D = {D:.2f}")
for lam in [500, 1000, 2000, 2416.73, 3000]:
    sk = Lasso(alpha=lam / len(x), tol=1e-10, max_iter=100000).fit(X, y).coef_[0]
    print(f"lambda {lam:8.2f}: formula {lasso_slope(S, D, lam):6.3f}   sklearn {sk:6.3f}")

S = 2416.73, D = 86.85
lambda   500.00: formula 22.071   sklearn 22.071
lambda  1000.00: formula 16.313   sklearn 16.313
lambda  2000.00: formula  4.799   sklearn  4.799
lambda  2416.73: formula  0.000   sklearn  0.000
lambda  3000.00: formula  0.000   sklearn  0.000
